In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [2]:
df = pd.read_csv("democleaned.csv")
df.head()

,YEAR,SERIAL,MONTH,CPSID,ASECFLAG,ASECWTH,REGION,PERNUM,CPSIDP,CPSIDV,...,OCC1990,IND1990,CLASSWKR,UHRSWORKT,EDUC,WKSWORK1,INCWAGE,RACE_REC,RACE_ETH,EDUCYRS
0,2000,2,3,19981202520100,1,974.38,New England Division,1,19981202520101,199812025201011,...,Subject instructors (HS/college),Colleges and universities,1,48,Doctorate degree,52,44600,White,White,21
1,2000,4,3,20000102304500,1,946.16,New England Division,1,20000102304501,200001023045011,...,"Salespersons, n.e.c.","Sporting goods, bicycles, and hobby stores",0,39,High school diploma or equivalent,52,20000,White,White,12
2,2000,6,3,20000203375200,1,1053.90,New England Division,1,20000203375201,200002033752011,...,Primary school teachers,Elementary and secondary schools,1,40,Master's degree,52,32000,White,White,18
3,2000,9,3,19990101648800,1,1053.39,New England Division,1,19990101648801,199901016488011,...,"Teachers , n.e.c.",Music stores,0,50,Bachelor's degree,52,5000,White,White,16
4,2000,11,3,20000105247100,1,861.97,New England Division,1,20000105247101,200001052471011,...,Registered nurses,Hospitals,0,50,Bachelor's degree,52,46000,White,White,16


In [3]:
df['RACE_ETH'].unique()

array(['White', 'Black', 'Asian', 'Native', 'Other'], dtype=object)

In [4]:
df['EXPERIENCE'] = df['AGE'] - df['EDUCYRS'] - 6
df['EXPERIENCE_SQ'] = df['EXPERIENCE'] ** 2

In [5]:
df.columns

Index(['YEAR', 'SERIAL', 'MONTH', 'CPSID', 'ASECFLAG', 'ASECWTH', 'REGION',
       'PERNUM', 'CPSIDP', 'CPSIDV', 'ASECWT', 'AGE', 'SEX', 'RACE', 'MARST',
       'NATIVITY', 'HISPAN', 'OCC1990', 'IND1990', 'CLASSWKR', 'UHRSWORKT',
       'EDUC', 'WKSWORK1', 'INCWAGE', 'RACE_REC', 'RACE_ETH', 'EDUCYRS',
       'EXPERIENCE', 'EXPERIENCE_SQ'],
      dtype='object')

In [6]:
df['HOURLY_WAGE'] = df['INCWAGE'] / (df['WKSWORK1'] * df['UHRSWORKT'])
lower = df['HOURLY_WAGE'].quantile(0.01)
upper = df['HOURLY_WAGE'].quantile(0.99)

df['HOURLY_WAGE'] = df['HOURLY_WAGE'].clip(lower=lower, upper=upper)


In [7]:
df['GOV_SECTOR'] = df['CLASSWKR'].astype(int)
df['RACE_ETH'] = np.where(df['HISPAN'] == 1, 'Hispanic', df['RACE'])

In [8]:
df['LNWAGE'] = np.log(df['HOURLY_WAGE'])

In [9]:
df.columns

Index(['YEAR', 'SERIAL', 'MONTH', 'CPSID', 'ASECFLAG', 'ASECWTH', 'REGION',
       'PERNUM', 'CPSIDP', 'CPSIDV', 'ASECWT', 'AGE', 'SEX', 'RACE', 'MARST',
       'NATIVITY', 'HISPAN', 'OCC1990', 'IND1990', 'CLASSWKR', 'UHRSWORKT',
       'EDUC', 'WKSWORK1', 'INCWAGE', 'RACE_REC', 'RACE_ETH', 'EDUCYRS',
       'EXPERIENCE', 'EXPERIENCE_SQ', 'HOURLY_WAGE', 'GOV_SECTOR', 'LNWAGE'],
      dtype='object')

In [10]:
df["RACE_ETH"].unique()

array(['100', 'Hispanic', '200', '650', '300', '803', '802', '651', '652',
       '801', '804', '830', '806', '810', '805', '809', '807', '820',
       '812', '814', '811', '808', '813', '816', '817', '818', '815',
       '819'], dtype=object)

In [11]:
df["GOV_SECTOR"].unique()

array([1, 0])

In [12]:
df.shape

(1402419, 32)

In [13]:
df.to_csv("CBSEDA.csv")

In [14]:
df["RACE_ETH"] = np.where(df["HISPAN"] == 1, "Hispanic", df["RACE_REC"])


In [15]:
formula = (
    'LNWAGE ~ EXPERIENCE + EXPERIENCE_SQ + C(EDUCYRS) + '       
    'C(SEX) + C(MARST) + C(NATIVITY) + '                         
    'C(RACE_ETH, Treatment(reference="White")) * '              
    'C(GOV_SECTOR, Treatment(reference=0)) + '                   
    'C(REGION) + C(YEAR)'                                        
)

print("Dependent Variable: ln(Hourly Wage)")
print("Reference Race: White | Reference Sector: Private")

model = smf.wls(formula=formula, data=df, weights=df['ASECWT'])
results = model.fit()

print(results.summary())

Dependent Variable: ln(Hourly Wage)
Reference Race: White | Reference Sector: Private
                            WLS Regression Results                            
Dep. Variable:                 LNWAGE   R-squared:                       0.351
Model:                            WLS   Adj. R-squared:                  0.351
Method:                 Least Squares   F-statistic:                 1.187e+04
Date:                Thu, 16 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:00:25   Log-Likelihood:            -1.4873e+06
No. Observations:             1402419   AIC:                         2.975e+06
Df Residuals:                 1402354   BIC:                         2.976e+06
Df Model:                          64                                         
Covariance Type:            nonrobust                                         
                                                                                                       coef    std err      